In [106]:
# libraries:
import json

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "config.py").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.genai import types
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.agents.context import Context
from google.adk.agents.readonly_context import ReadonlyContext
from google.adk.tools import FunctionTool
from google.adk.tools.agent_tool import AgentTool
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest


In [107]:
# system instructions:
from core.instructions.planner import PLANNER_PROMPT
from core.instructions.blocks.base import BASE_BLOCK
from core.instructions.blocks.onboarding import ONBOARDING_BLOCK


In [108]:
# structured output:
from core.plan_schema import Plan


In [109]:
# configuration:
from core.config import APP_NAME, USER_ID, SESSION_ID, KNOWLEDGE_SPACE


In [110]:
gemma_4_26B = LiteLlm(model="openai//models/gemma-4-26b-a4b-it", api_base="http://127.0.0.1:18001/v1", api_key="null")

In [111]:
def log_before_agent(callback_context: CallbackContext):
    print("Agent start:", callback_context.agent_name)
    return None

def log_after_tool(tool, args, tool_context, tool_response):
    print("Tool finished:", tool.name, tool_response)
    return None

def log_before_model(callback_context: CallbackContext, llm_request: LlmRequest):
    print("Before model:", callback_context.agent_name)
    return None

In [112]:
def retrieve_topic_by_id(context: Context, topic_id: str) -> str:
    """
    Retrieve one topic from the local knowledge graph by exact topic id.
    Valid topic ids: supervised-learning, data-splits, overfitting.
    """
    for topic in KNOWLEDGE_SPACE.get("topics", []):
        if topic.get("id") == topic_id:
            topic_json = json.dumps(topic, ensure_ascii=False, indent=2)
            context.state["active_topic_id"] = topic["id"]
            context.state["active_topic_title"] = topic.get("title", topic["id"])
            context.state["active_topic_json"] = topic_json
            return f"Stored topic '{topic['id']}' in shared state for planning."
    return f"Unknown topic_id: {topic_id}. Valid topic ids: supervised-learning, data-splits, overfitting"

retrieve_topic_by_id_tool = FunctionTool(retrieve_topic_by_id)


In [113]:
def planner_instruction(context: ReadonlyContext) -> str:
    active_topic_json = context.state.get("active_topic_json")

    if active_topic_json:
        return PLANNER_PROMPT + "# Retrieved Topic Context: " + str(active_topic_json)
    
    return PLANNER_PROMPT

planner_agent = Agent(
    model=gemma_4_26B,
    name="Planner",
    description="Planner agent.",
    instruction=planner_instruction,
    output_schema=Plan,
    generate_content_config=types.GenerateContentConfig(temperature=0),
    before_agent_callback=log_before_agent,
    before_model_callback=log_before_model,
    after_tool_callback=log_after_tool,
)

In [114]:
retriever_agent = Agent(
    model=gemma_4_26B,
    name="Retriever",
    description="Retriever agent.",
    instruction=(
        "You are a topic retriever.\n"
        "Use `retrieve_topic_by_id` to load exactly one topic into shared session state.\n"
        "Valid topic ids: supervised-learning, data-splits, overfitting.\n"
        "If the learner asks where to start, use `supervised-learning`.\n"
        "After the tool call, return one short sentence naming the stored topic id.\n"
        "Do not return the topic JSON."
    ),
    tools=[retrieve_topic_by_id_tool],
    before_agent_callback=log_before_agent,
    before_model_callback=log_before_model,
    after_tool_callback=log_after_tool,
)

In [115]:
assistant_agent = Agent(
    model=gemma_4_26B,
    name="Assistant",
    description="Assistant agent.",
    instruction=(
        BASE_BLOCK
        + "\n\n# Testing Mode\n"
        + "- Use `Retriever` to store the selected topic in shared session state.\n"
        + "- Valid topic ids are: `supervised-learning`, `data-splits`, `overfitting`.\n"
        + "- If the learner asks where to start, call `Retriever` for `supervised-learning` first.\n"
        + "- Call `Planner` only after a topic is stored in shared session state.\n"
        + "- Do not paste the full retrieved topic into your reply."
    ),
    tools=[
        AgentTool(agent=planner_agent,), # skip_summarization=True
        AgentTool(agent=retriever_agent), 
    ],
    before_agent_callback=log_before_agent,
    before_model_callback=log_before_model,
    after_tool_callback=log_after_tool,
)

session_service = InMemorySessionService()

await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

runner = Runner(
    agent=assistant_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

In [116]:
async def send_message(text: str) -> str | None:
    message = types.Content(
        role="user",
        parts=[types.Part(text=text)],
    )

    final_text = None

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [p.text for p in event.content.parts if getattr(p, "text", None)]
            final_text = "\n".join(texts)

    return final_text


In [117]:
await send_message("Hello! I am Eduard.")

Agent start: Assistant
Before model: Assistant


'Hello, Eduard. It is nice to meet you.\n\nI am here to help you navigate the field of machine learning. Whether you are just starting out or looking to deepen your understanding of specific concepts, we can work through the material together at a pace that feels right for you.\n\nTo get us started, is there a particular area of machine learning you are interested in exploring, or would you like me to suggest a starting point?'

In [118]:
await send_message("""
I'd say my Python is at a basic to intermediate level. 
I'm comfortable with things like variables, loops, functions, pandas, and some simple data analysis, 
but I still need more practice with more advanced topics.
For math, I have some familiarity, but I wouldn't say I'm very strong yet. 
I know the basics of probability and statistics better than linear algebra and calculus. With linear algebra and calculus, 
I've seen some concepts before, but I would probably need to review them properly and understand how they apply in data science.
When learning something new, I prefer to start with the high-level intuition and real-world applications first. 
That helps me understand why the concept matters. After that, I find it easier to go deeper into the mathematical foundations and theory.
""")

Agent start: Assistant
Before model: Assistant


'Thank you for sharing that, Eduard. Having a solid grasp of `pandas` and basic data analysis is a wonderful foundation; those are the tools you will use constantly to prepare and inspect data.\n\nYour preference for starting with intuition and real-world utility before moving into the mathematics is a very effective way to build durable mental models. We can certainly follow that rhythm: building a conceptual "hook" first, and then filling in the mathematical structure once you see why it is necessary.\n\nSince you mentioned you are comfortable with probability and statistics but would like to see how linear algebra and calculus apply to data science, we could start with the core pillars of the field.\n\nWould you like to begin with an overview of **supervised learning**, or is there a specific mathematical concept you\'d like to see contextualized within machine learning first?'

In [119]:
await send_message("Practical foundation")

Agent start: Assistant
Before model: Assistant
Agent start: Retriever
Before model: Retriever
Tool finished: retrieve_topic_by_id Stored topic 'supervised-learning' in shared state for planning.
Before model: Retriever
Tool finished: Retriever Stored topic id: supervised-learning.
Before model: Assistant
Agent start: Planner
Before model: Planner
Tool finished: Planner {'topic': 'What supervised learning is', 'topic_id': 'supervised-learning', 'title': 'Foundations of Supervised Learning', 'strategy': {'intent': 'Introduce the core concept of supervised learning by mapping statistical intuition to machine learning terminology.', 'emphasis': 'Intuition-first approach focusing on the relationship between inputs and targets.', 'out_of_scope': ['Deep learning architectures', 'Specific optimization algorithms', 'Manual implementation of gradient descent']}, 'description': 'Learn how models use labeled data to map input features to specific target outputs.', 'learning_objective': 'Distinguis

'That sounds like a wise approach. To build a practical foundation, we should start with the most common framework used in the field: **supervised learning**.\n\nIn essence, supervised learning is the process of teaching a machine by showing it examples that already have the "right answer" attached. \n\nImagine you are teaching a child to identify different types of fruit. You don\'t just show them a pile of fruit; you pick up an apple and say, "This is an apple." Then you pick up a banana and say, "This is a banana." Eventually, when you show them a new piece of fruit they haven\'t seen before, they can use the patterns they\'ve learned—like shape, color, and texture—to make an educated guess.\n\nIn machine learning, we call those characteristics (color, shape, size) **features**, and the "right answer" (apple, banana) the **target** or **label**.\n\nBefore we look at the different ways we can use this, does the distinction between the **features** (the inputs) and the **target** (the

In [120]:
await send_message("You can suggest")

Agent start: Assistant
Before model: Assistant


"To ensure we are on the same page before moving forward, I'll offer a quick way to test that intuition.\n\nImagine you are building a model to predict the price of a house based on its square footage, the number of bedrooms, and its location.\n\nIn this scenario, which of those pieces of information would you consider to be the **features**, and which would be the **target**?"